# Lab 2: Word Representations — Solutions
### Questions 1–4: BoW, TF-IDF, GloVe, Word2Vec

---
## Question 1: Bag-of-Words on Amazon Reviews (Last 50,000 rows)

In [ ]:
# Install datasets library for HuggingFace
!pip install datasets -q

In [ ]:
import re
import string
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# ── Load Amazon Reviews Polarity (train split) ──────────────────────────────
print("Loading Amazon Customer-Reviews Polarity dataset...")
dataset = load_dataset("amazon_polarity", split="train")
df_full = dataset.to_pandas()
print(f"Full train set shape: {df_full.shape}")
print(f"Columns: {df_full.columns.tolist()}")

In [ ]:
# ── Take last 50,000 rows & drop 'title' ────────────────────────────────────
df = df_full.tail(50000).copy()
df = df.drop(columns=["title"])
df = df.reset_index(drop=True)
print(f"Subset shape after dropping 'title': {df.shape}")
df.head(3)

In [ ]:
# ── Preprocessing: lowercase + remove punctuation ───────────────────────────
def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text

df["clean_content"] = df["content"].apply(preprocess)
print("Sample preprocessed text:")
print(df["clean_content"].iloc[0])

In [ ]:
# ── CountVectorizer: Bag-of-Words ────────────────────────────────────────────
bow_vectorizer = CountVectorizer()
X_bow = bow_vectorizer.fit_transform(df["clean_content"])

print("=" * 50)
print(f"BoW Feature Matrix Dimensions: {X_bow.shape}")
print(f"  Rows    (documents) : {X_bow.shape[0]}")
print(f"  Columns (features)  : {X_bow.shape[1]}")
print("=" * 50)
print(f"\nMatrix type : {type(X_bow)}")
print(f"Stored non-zero elements: {X_bow.nnz}")

---
## Question 2: TF-IDF Embeddings — Top 25 Words

In [ ]:
# ── TF-IDF Vectorizer ────────────────────────────────────────────────────────
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df["clean_content"])

print(f"TF-IDF Feature Matrix Dimensions: {X_tfidf.shape}")

In [ ]:
# ── Top 25 words by highest mean TF-IDF score ────────────────────────────────
feature_names = tfidf_vectorizer.get_feature_names_out()

# Compute mean TF-IDF score across all documents for each word
mean_tfidf_scores = np.array(X_tfidf.mean(axis=0)).flatten()

# Build a sorted DataFrame
tfidf_df = pd.DataFrame({
    "word": feature_names,
    "mean_tfidf": mean_tfidf_scores
})
top25 = tfidf_df.nlargest(25, "mean_tfidf").reset_index(drop=True)
top25.index += 1  # rank from 1

print("Top 25 words with highest mean TF-IDF scores:")
print(top25.to_string())

In [ ]:
# ── Optional: visualise with a bar chart ────────────────────────────────────
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.barh(top25["word"][::-1], top25["mean_tfidf"][::-1], color="steelblue")
plt.xlabel("Mean TF-IDF Score")
plt.title("Top 25 Words by Mean TF-IDF Score (Amazon Reviews)")
plt.tight_layout()
plt.show()

---
## Question 3: GloVe Embeddings — Analogy Task (Paris - France + Italy ≈ Rome)

In [ ]:
# ── Download GloVe 6B 100d ───────────────────────────────────────────────────
import os
import zipfile

GLOVE_ZIP  = "glove.6B.zip"
GLOVE_FILE = "glove.6B.100d.txt"

if not os.path.exists(GLOVE_FILE):
    print("Downloading GloVe 6B zip (~822 MB) — this may take a few minutes…")
    !wget -q --show-progress https://nlp.stanford.edu/data/glove.6B.zip -O {GLOVE_ZIP}
    with zipfile.ZipFile(GLOVE_ZIP, "r") as z:
        z.extract(GLOVE_FILE)
    print("Extraction complete.")
else:
    print(f"{GLOVE_FILE} already exists — skipping download.")

In [ ]:
# ── Load GloVe vectors ───────────────────────────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

def load_glove(filepath):
    print(f"Loading GloVe from {filepath}…")
    word_vectors = {}
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.split()
            word   = parts[0]
            vector = np.array(parts[1:], dtype="float32")
            word_vectors[word] = vector
    print(f"Loaded {len(word_vectors):,} word vectors (dim={len(next(iter(word_vectors.values())))}).")
    return word_vectors

glove = load_glove(GLOVE_FILE)

In [ ]:
# ── Analogy: Paris - France + Italy ─────────────────────────────────────────
def find_analogy(glove_model, word_a, word_b, word_c, topn=5):
    """
    Analogy: word_a - word_b + word_c = ?
    E.g.  Paris  - France  + Italy  = Rome
    """
    for w in [word_a, word_b, word_c]:
        if w not in glove_model:
            raise KeyError(f"'{w}' not found in GloVe vocabulary.")

    target_vec = (glove_model[word_a]
                  - glove_model[word_b]
                  + glove_model[word_c])

    exclude = {word_a, word_b, word_c}

    # Build matrix of all word vectors (only once ideally — fine for demo)
    words   = [w for w in glove_model if w not in exclude]
    vectors = np.array([glove_model[w] for w in words])

    sims = cosine_similarity([target_vec], vectors)[0]
    top_indices = np.argsort(sims)[::-1][:topn]

    results = [(words[i], sims[i]) for i in top_indices]
    return results, target_vec


print("Analogy: Paris − France + Italy = ?")
print("-" * 45)
results, analogy_vec = find_analogy(glove, "paris", "france", "italy", topn=10)

for rank, (word, score) in enumerate(results, 1):
    marker = " ← ✓  (Expected answer!)" if word == "rome" else ""
    print(f"  {rank:2d}. {word:<20s}  cosine = {score:.4f}{marker}")

top_word = results[0][0]
print(f"\nClosest word to the analogy vector: '{top_word}'")

rome_rank = next((i+1 for i, (w, _) in enumerate(results) if w == "rome"), None)
if rome_rank:
    print(f"'rome' appears at rank {rome_rank} in the top-10 results. ✓")
else:
    print("'rome' was not in the top-10 results.")

In [ ]:
# ── Cosine similarity of analogy result with 'rome' ─────────────────────────
rome_vec = glove["rome"]
sim_to_rome = cosine_similarity([analogy_vec], [rome_vec])[0][0]
print(f"Cosine similarity between analogy vector and 'rome': {sim_to_rome:.4f}")

---
## Question 4: Word2Vec (Skip-gram & CBOW) on Brown Corpus — Analogy Task

In [ ]:
# ── Install / import gensim & NLTK ──────────────────────────────────────────
!pip install gensim -q

import nltk
nltk.download("brown")
nltk.download("punkt")

from nltk.corpus import brown
from gensim.models import Word2Vec

In [ ]:
# ── Load & preprocess Brown corpus ──────────────────────────────────────────
print("Loading Brown corpus…")
sentences_raw = brown.sents()

# Tokenize & lowercase
sentences = [[token.lower() for token in sent] for sent in sentences_raw]

print(f"Total sentences : {len(sentences):,}")
print(f"Sample sentence : {sentences[0]}")

In [ ]:
# ── Train Skip-gram model (sg=1) ─────────────────────────────────────────────
print("Training Skip-gram Word2Vec…")
model_sg = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,          # 1 = Skip-gram
    epochs=10,
    workers=4,
    seed=42
)
print(f"Skip-gram vocabulary size: {len(model_sg.wv):,}")

In [ ]:
# ── Train CBOW model (sg=0) ──────────────────────────────────────────────────
print("Training CBOW Word2Vec…")
model_cbow = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0,          # 0 = CBOW
    epochs=10,
    workers=4,
    seed=42
)
print(f"CBOW vocabulary size: {len(model_cbow.wv):,}")

In [ ]:
# ── Analogy Evaluation ────────────────────────────────────────────────────────
# The Brown corpus is small (~57k sentences), so capital-city pairs like
# paris/france/italy/rome may be missing from its vocabulary.
# We first try the Q3 analogy; if any word is absent we fall back to a
# corpus-appropriate analogy.

def w2v_analogy(model, pos1, neg1, pos2, topn=10, label=""):
    """
    Analogy: pos1 - neg1 + pos2 = ?
    Uses gensim's most_similar with positive/negative lists.
    """
    print(f"\n{'='*55}")
    print(f" Model  : {label}")
    print(f" Analogy: {pos1} - {neg1} + {pos2} = ?")
    print(f"{'='*55}")

    # Check vocab
    missing = [w for w in [pos1, neg1, pos2] if w not in model.wv]
    if missing:
        print(f"  ✗ Words not in W2V vocabulary: {missing}")
        print("  → Skipping this analogy for this model.")
        return None

    results = model.wv.most_similar(
        positive=[pos1, pos2],
        negative=[neg1],
        topn=topn
    )
    for rank, (word, score) in enumerate(results, 1):
        print(f"  {rank:2d}. {word:<20s}  cosine = {score:.4f}")
    return results


# ── Try Q3 analogy first ─────────────────────────────────────────────────────
ANALOGY = ("paris", "france", "italy")   # expected: rome

res_sg   = w2v_analogy(model_sg,   *ANALOGY, label="Skip-gram")
res_cbow = w2v_analogy(model_cbow, *ANALOGY, label="CBOW")

In [ ]:
# ── Fallback analogy (corpus-appropriate) if words are absent ────────────────
# Brown corpus is a general American English corpus from the 1960s.
# A reliable analogy: 'man' - 'king' + 'queen' ≈ 'woman'

FALLBACK = ("queen", "king", "man")   # expected: woman

if res_sg is None:
    print("\n[Fallback analogy for Skip-gram]")
    w2v_analogy(model_sg, *FALLBACK, label="Skip-gram (fallback)")

if res_cbow is None:
    print("\n[Fallback analogy for CBOW]")
    w2v_analogy(model_cbow, *FALLBACK, label="CBOW (fallback)")

In [ ]:
# ── Quick sanity check: most similar words ───────────────────────────────────
for probe in ["man", "government", "city"]:
    if probe in model_sg.wv:
        sim = model_sg.wv.most_similar(probe, topn=5)
        print(f"\nSkip-gram — most similar to '{probe}':")
        for w, s in sim:
            print(f"  {w:<20s} {s:.4f}")

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print(" SUMMARY")
print("="*60)
print(f" Q1  BoW matrix dimensions       : {X_bow.shape}")
print(f" Q2  TF-IDF matrix dimensions    : {X_tfidf.shape}")
print(f"     Top-1 word (highest TF-IDF) : {top25['word'].iloc[0]}")
print(f" Q3  GloVe analogy top-1 result  : {results[0][0]}")
print(f"     Cosine sim to 'rome'        : {sim_to_rome:.4f}")
print(f" Q4  Skip-gram vocab size        : {len(model_sg.wv):,}")
print(f"     CBOW vocab size             : {len(model_cbow.wv):,}")
print("="*60)